This implements advisory agent

In [21]:
import investor_profiles
import stock_universe
import disclosure_snippets
import numpy as np

In [2]:
RISK_PROFILES = [
{"risk_tolerance": "Conservative", "distribution_strategy": "equal-weight", "stocks_list": ["PAYBOND", "PAYGOLD", "PAYRETAIL"]},
{"risk_tolerance": "Moderate", "distribution_strategy": "equal-weight", "stocks_list": ["PAYRETAIL", "PAYINFRA", "PAYGOLD"]},
{"risk_tolerance": "Aggressive", "distribution_strategy": "equal-weight", "stocks_list": ["PAYTECH", "PAYFIN", "PAYINFRA"]}
]

In [34]:
# Defines the tools

def get_stock_data(ticker: str):
  return stock_universe.STOCK_UNIVERSE.get(ticker), stock_universe.MARKET_RETURN, stock_universe.RISK_FREE_RATE

def calculate_expected_return_stock(Rm: float, Rf: float, beta: float):
  return Rf + beta * (Rm - Rf)

def calculate_return_portfolio(weights: np.array, stocks: np.array, Rm: float, Rf: float, correlation: float):
  weighted_return = 0
  for i in range(len(stocks)):
    weighted_return += calculate_expected_return_stock(Rm, Rf, stocks[i][0].get('beta')) * weights[i]
    print("Current weighted return: ", weighted_return)
  return weighted_return

def calculate_variance_portfolio(weights: np.array, stocks: np.array, correlation: float):
  # Var(R_p) = Σᵢ wᵢ²σᵢ² + 2·Σ_{i<j} wᵢwⱼ·Cov(Rᵢ,Rⱼ), with Cov(Rᵢ,Rⱼ) = ρ·σᵢ·σⱼ
  # correlation ρ = 0.3 for every pair of the three tickers. Convert variance to portfolio standard deviation
  variance = 0
  # Calculate Σᵢ wᵢ²σᵢ²
  for i in range(len(stocks)):
    sigma_i = stocks[i][0].get('std_dev')
    variance += (weights[i]**2) * (sigma_i**2)

  # Calculate 2·Σ_{i<j} wᵢwⱼ·Cov(Rᵢ,Rⱼ)
  for i in range(len(stocks)):
    for j in range(i + 1, len(stocks)):
      sigma_i = stocks[i][0].get('std_dev')
      sigma_j = stocks[j][0].get('std_dev')
      covariance = correlation * sigma_i * sigma_j
      variance += 2 * weights[i] * weights[j] * covariance
      print("Current variance: ", variance)
  return np.sqrt(variance)

In [35]:
stock_data, Rm, Rf = get_stock_data("PAYBOND")
print("Stock Data -> ", Rm, Rf, stock_data)
print("Return for stock : ", f"{calculate_expected_return_stock(Rm, Rf, stock_data.get('beta')):.2}")
weights=np.array([1/3, 1/3, 1/3])
stocks=np.array([get_stock_data("PAYBOND"), get_stock_data("PAYGOLD"), get_stock_data("PAYRETAIL")])
portfolio_return = calculate_return_portfolio(weights, stocks, Rm, Rf, 0.3)
portfolio_std_dev = calculate_variance_portfolio(weights=weights, stocks=stocks, correlation=0.3)
print("Portfolio return: ", f"{portfolio_return:.2}")
print("Portfolio variance: ", f"{portfolio_std_dev:.2}")

Stock Data ->  0.13 0.07 {'beta': 0.05, 'analyst_expected_return': 0.065, 'std_dev': 0.04}
Return for stock :  0.073
Current weighted return:  0.024333333333333335
Current weighted return:  0.051666666666666666
Current weighted return:  0.092
Current variance:  0.005308888888888889
Current variance:  0.005762222222222222
Current variance:  0.0071222222222222225
Portfolio return:  0.092
Portfolio variance:  0.084
